# ohlcv-lab: итоговый отчёт

Пять монет Bybit (USDT perpetual), часовые свечи за **ровно три года: 2023-10-01 — 2026-10-01**
(правая граница не включается). Период зафиксирован в `ohlcv_lab.report`: данные докачиваются при
каждом запуске, и без фиксированного периода цифры менялись бы каждую неделю.

Ноутбук только вызывает функции из `src/`; те же функции рисуют картинки для README
(`ohlcv-report`), поэтому отчёт и README не расходятся.

Вопросы:
1. Насколько качественные данные?
2. Сколько принесли монеты и какой ценой (волатильность, просадки)?
3. Насколько хвосты толще нормальных и что это значит для оценки риска?
4. Есть ли диверсификация внутри крипты?

In [ ]:
%matplotlib inline
import json

import pandas as pd

from ohlcv_lab import report

CLEAN = "../data/clean"  # путь относительно папки notebooks/

closes = report.load_period(clean_dir=CLEAN)  # упадёт, если период покрыт не полностью
closes.shape

## 1. Качество данных

Отчёт о качестве пишет `ohlcv-clean`. Пропуски не интерполируются, а помечаются; незакрытая на момент
выгрузки свеча отбрасывается; экстремальные свечи помечаются по скользящим статистикам только из
прошлого (без look-ahead).

In [ ]:
quality = pd.DataFrame(json.load(open(f"{CLEAN}/quality_report.json")))
quality["gaps"] = quality["gaps"].str.len()
cols = ["rows", "first", "last", "duplicates", "missing", "gaps", "unclosed_dropped"]
quality.set_index(["symbol", "interval"])[cols + ["extreme_candles"]]

## 2. Доходность и её цена

Шкала логарифмическая: одинаковые движения в процентах выглядят одинаково, а не растут вместе с ценой.

In [ ]:
report.plot_growth(closes)

In [ ]:
report.plot_vol(closes)

Волатильность кластеризуется: спокойные месяцы сменяются бурными, и у всех монет одновременно —
это общий рыночный фактор.

In [ ]:
report.plot_drawdowns(closes)

## 3. Толстые хвосты

`var_1%` — исторический квантиль: в 1% часов потеря была больше. `var_1%_normal` — то же по
нормальной модели с теми же средним и σ. `tail_share` — доля часов с движением > 4σ (у нормального
распределения 0.006%).

In [ ]:
table = report.summary_table(closes)
table.style.format(
    {
        "ann_vol": "{:.0%}",
        "excess_kurtosis": "{:.1f}",
        "var_1%": "{:.2%}",
        "var_1%_normal": "{:.2%}",
        "max_drawdown": "{:.0%}",
        "tail_share": "{:.2%}",
    }
)

In [ ]:
report.plot_tails(closes)

## 4. Корреляции

In [ ]:
report.plot_corr(closes)

## Выводы

- **Данные:** полная сетка без пропусков и дублей на всём периоде; незакрытые свечи отброшены.
- **Риск разный в 2 раза:** годовая волатильность от ~47% (BTC) до ~92% (DOGE), и она меняется во
  времени — одна цифра σ за 3 года описывает риск грубо.
- **Просадки глубокие:** от −54% (BTC) до −86% (DOGE) от максимума.
- **Хвосты толстые:** эксцесс 9–28, движений > 4σ в ~100 раз больше, чем ждёт нормальная модель;
  нормальный VaR 1% занижает исторический на 20–30%. Для риска — исторические оценки.
- **Диверсификации почти нет:** корреляции часовых доходностей 0.6–0.8, в обвал падает всё вместе.

Ограничения: один период (3 года) и одна биржа; средняя доходность за час в ~100 раз меньше σ, поэтому
любые выводы о доходности (Sharpe) сильно зависят от выбранного периода.